# Generating biological time courses with a PyTorch GAN

The experiment learns the distribution of stimulated single-cell trajectories. It is intentionally unconditional: generated samples reproduce population-level variation but are not tied to a treatment label.

## 1. Setup and reproducibility

Install the repository with `python -m pip install -e ".[torch]"`. Model definitions and optimization are imported from the project package.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from deeplearning_examples.io import load_stimulation_timecourses
from pytorch_cdgan.model import Discriminator, Generator, count_parameters, initialize_weights
from pytorch_cdgan.training import GanTrainingConfig, train_gan

RANDOM_SEED = 32
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Load and inspect trajectories

The shared loader combines the 2013 and 2014 stimulation experiments. Each row is one cell and each column one timepoint.

In [ ]:
trajectories = load_stimulation_timecourses()
print(f"trajectories: {trajectories.shape}")
print(f"range: {trajectories.min():.3f} to {trajectories.max():.3f}")

selection = np.random.default_rng(RANDOM_SEED).choice(len(trajectories), 20, replace=False)
plt.figure(figsize=(9, 3))
plt.plot(trajectories[selection].T, alpha=0.35)
plt.xlabel("time index")
plt.ylabel("signal")
plt.title("Observed stimulation trajectories")
plt.tight_layout()

## 3. Build generator and discriminator

The generator maps a 16-dimensional noise vector to a smooth one-dimensional signal. The discriminator scores complete trajectories rather than isolated timepoints.

In [ ]:
generator = Generator(noise_size=16, output_length=trajectories.shape[1])
discriminator = Discriminator(input_length=trajectories.shape[1])
generator.apply(initialize_weights)
discriminator.apply(initialize_weights)
print(f"generator parameters:     {count_parameters(generator):,}")
print(f"discriminator parameters: {count_parameters(discriminator):,}")

### Initial scale check

Before training, compare generated and observed ranges. A severe scale mismatch can make the discriminator's initial task trivial.

In [ ]:
generator.eval()
with torch.no_grad():
    initial_samples = generator(torch.randn(3, 16)).squeeze(1).numpy()
plt.figure(figsize=(8, 3))
plt.plot(initial_samples.T)
plt.title("Untrained generator output")
plt.tight_layout()

## 4. Adversarial training

`train_gan` performs separate discriminator and generator updates, records scalar diagnostics and periodically stores generated samples.

In [ ]:
history = train_gan(
    generator,
    discriminator,
    trajectories,
    config=GanTrainingConfig(
        epochs=2_000,
        batch_size=256,
        snapshot_every=100,
        report_every=50,
    ),
    device=device,
)

## 5. Training diagnostics

Discriminator scores near 0.5 indicate uncertainty, but not necessarily high-quality samples. Loss curves and trajectory morphology must therefore be considered together.

In [ ]:
epochs = np.arange(1, len(history.generator_loss) + 1)
figure, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(epochs, history.discriminator_real_loss, label="D real")
axes[0].plot(epochs, history.discriminator_fake_loss, label="D fake")
axes[0].plot(epochs, history.generator_loss, label="G")
axes[0].set(title="Adversarial losses", xlabel="epoch")
axes[0].legend()
axes[1].plot(epochs, history.real_score, label="real")
axes[1].plot(epochs, history.fake_score, label="fake")
axes[1].axhline(0.5, color="black", linestyle=":")
axes[1].set(title="Discriminator scores", xlabel="epoch", ylabel="mean score")
axes[1].legend()
figure.tight_layout()

## 6. Generated trajectories over training

Snapshots reveal whether the generator learns plausible dynamics or collapses to a small set of shapes.

In [ ]:
figure, axes = plt.subplots(min(6, len(history.snapshots)), 1, figsize=(9, 10), sharex=True)
axes = np.atleast_1d(axes)
for axis, snapshot_index in zip(axes, np.linspace(0, len(history.snapshots)-1, len(axes), dtype=int)):
    axis.plot(history.snapshots[snapshot_index].squeeze(1).T, alpha=0.55)
    axis.set_ylabel(f"#{snapshot_index}")
axes[-1].set_xlabel("time index")
figure.suptitle("Generator snapshots")
figure.tight_layout()

## 7. Interpretation

Useful follow-up diagnostics include nearest-neighbor checks against the training set, diversity metrics and comparisons of burst frequency, amplitude and duration. A plausible-looking mean trajectory alone is insufficient evidence that the generator learned the biological distribution.